# Transfer Learning - walidacja krzyżowa (29-fold CV)

Ten notebook implementuje **transfer learning** dla modelu DeepMReye na danych resting-state.

In [ ]:
import os
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
import tensorflow as tf
tf.keras.backend.clear_session()
import pandas as pd
from deepmreye import analyse, architecture, preprocess, train
from deepmreye.util import data_generator, model_opts, util
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.model_selection import KFold
from smooth_loss import create_model_with_smoothl1, train_model_with_smoothl1

gpus = tf.config.experimental.list_physical_devices("GPU")
tf.config.experimental.set_memory_growth(gpus[0], True)

## Funkcje pomocnicze

`create_holdout_generators` opakowuje generator danych DeepMReye.
Przyjmuje jawne listy ścieżek i zwraca generatory treningowe, testowe oraz per-podmiot.

In [ ]:
def create_holdout_generators(datasets=None, train_split=0.6, train_list=None, test_list=None, **args):
    if train_list is not None and test_list is not None:
        full_training_list = train_list
        full_testing_list = test_list
    else:
        full_training_list, full_testing_list = list(), list()
        for fn_data in datasets:
            this_file_list = [fn_data + p for p in os.listdir(fn_data)]
            np.random.shuffle(this_file_list)
            split_idx = int(train_split * len(this_file_list))
            full_training_list.extend(this_file_list[:split_idx])
            full_testing_list.extend(this_file_list[split_idx:])

    (training_generator, testing_generator,
     single_testing_generators, single_testing_names,
     single_training_generators, single_training_names,
    ) = data_generator.create_generators(full_training_list, full_testing_list, **args)

    return (training_generator, testing_generator,
            single_testing_generators, single_testing_names,
            single_training_generators, single_training_names,
            full_testing_list, full_training_list)

## Wczytywanie danych

Wczytywany jest stały podział 23/6 podmiotów zapisany w plikach `.txt`.
Następnie oba zbiory są łączone i wszystkie 29 podmiotów trafia do puli CV,
a podział na fold treningowy/walidacyjny jest kontrolowany przez `KFold`.

In [ ]:
train_list_rs = np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/train_list_rs.txt', dtype=str)
test_list_rs  = np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/test_list_rs.txt',  dtype=str)

all_samples = np.concatenate([train_list_rs, test_list_rs])
print(f"Łączna liczba podmiotów do CV: {len(all_samples)}")

## Konfiguracja modelu i ścieżka do wag

Hiperparametry są dobrane pod fine-tuning przy zamrożonym backbone:
- **Niski learning rate** (6.3e-06) - żeby nie niszczyć wytrenowanych wag
- **MC Dropout** - regularizacja przez losowe wyłączanie neuronów
- **Małe kroki** (`steps_per_epoch=10`) - zbiór resting-state jest mały

In [ ]:
opts = model_opts.get_opts()
opts["epochs"] = 60
opts["lr"] = 6.307150285007128e-06
opts["gaussian_noise"] = 0.01
opts["dropout_rate"] = 0.0
opts["steps_per_epoch"] = 10
opts["validation_steps"] = 2
opts["num_fc"] = 1024
opts["mc_dropout"] = True
opts["loss_pearson"] = 0.01
opts["smooth_l1_delta"] = 0.03

cv_opts = opts.copy()

WEIGHTS_PATH = "/mnt/compneuro/deepmreye_finetuning/Zosia/modelinference_both_eyes_RS_mirror.h5"

## Pętla treningowa — 29-fold CV

Dla każdego z 29 foldów:
1. Podzielono 29 podmiotów na 28 treningowych i 1 walidacyjny
2. Załadowano wagi bazowe i zamrożono backbone (`freeze_backbone=True`)
3. Dotrenowano warstwy gęste przez maksymalnie 60 epok
4. Zapisano wyniki (best epoch, val loss, Pearson r) dla tego foldu

In [ ]:
kf = KFold(n_splits=29, shuffle=True, random_state=42)

fold_results = []
histories = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(all_samples)):
    print(f"FOLD {fold_idx + 1} / 29 - train: {len(train_idx)}, val: {len(val_idx)}")

    fold_train = all_samples[train_idx]
    fold_val = all_samples[val_idx]

    tf.keras.backend.clear_session()

    generators = data_generator.create_generators(
        full_training_list=fold_train,
        full_testing_list=fold_val,
        batch_size=cv_opts["batch_size"],
    )
    generators = (*generators, fold_train, fold_val)

    model, model_inf, history = train_model_with_smoothl1(
        dataset=f"cv_fold_{fold_idx + 1}",
        generators=generators,
        opts=cv_opts,
        is_resting_state=True,
        save=True,
        model_path=f"./cv_models/fold_{fold_idx + 1}/",
        pretrained_weights=WEIGHTS_PATH,
        freeze_backbone=True,
        verbose=1,
        use_multiprocessing=False,
        workers=1,
    )

    histories.append(history)

    best_epoch = np.argmin(history.history["val_smoothl1_loss"])
    fold_results.append({
        "fold": fold_idx + 1,
        "best_epoch": best_epoch + 1,
        "val_smoothl1_loss": min(history.history["val_smoothl1_loss"]),
        "val_pearson_r": history.history["val_pearson_r"][best_epoch],
        "val_subjects": list(fold_val),
    })
    print(f" loss={fold_results[-1]['val_smoothl1_loss']:.4f}, "
          f"pearson_r={fold_results[-1]['val_pearson_r']:.4f} (epoch {best_epoch+1})")

## Podsumowanie wyników CV

Wyświetlenie wyników per fold oraz średniej i odchylenia standardowego.
Wyniki są zapisywane do pliku CSV.

In [ ]:
losses = [r["val_smoothl1_loss"] for r in fold_results]
pearsons = [r["val_pearson_r"] for r in fold_results]

print(f"{'Fold':<8} {'Best Epoch':<12} {'Val Loss':<14} {'Pearson r':<10}")
for r in fold_results:
    print(f"  {r['fold']:<8} {r['best_epoch']:<12} {r['val_smoothl1_loss']:<14.4f} {r['val_pearson_r']:<10.4f}")
print(f"  {'Mean':<8} {np.mean([r['best_epoch'] for r in fold_results]):<12.1f} "
      f"{np.mean(losses):<14.4f} {np.mean(pearsons):<10.4f}")
print(f"  {'Std':<8} {np.std([r['best_epoch'] for r in fold_results]):<12.1f} "
      f"{np.std(losses):<14.4f} {np.std(pearsons):<10.4f}")

df = pd.DataFrame([{k: v for k, v in r.items() if k != "val_subjects"} for r in fold_results])
df.to_csv("./cv_models/cv_results_summary.csv", index=False)
print("\nWyniki zapisane do: ./cv_models/cv_results_summary.csv")

## Wizualizacje

Poniższe wykresy służą do analizy jakości i stabilności transfer learningu:
1. **Krzywe straty per fold** - czy trening przebiegał zdrowo w każdym foldzie
2. **Pearson r i strata per fold** - rozkład wyników między podmiotami
3. **Overfitting check** - luka między stratą treningową a walidacyjną
4. **Pearson r w czasie epok** - czy wszystkie foldy zbiegają podobnie

In [ ]:
N_FOLDS = len(fold_results)
FOLD_COLORS = [cm.tab20(i / N_FOLDS) for i in range(N_FOLDS)]

### Wykres 1 - Krzywe straty per fold

Linia ciągła = strata treningowa, linia przerywana = strata walidacyjna.
Pionowa kropkowana linia = najlepsza epoka (minimalna strata walidacyjna).
Duża rozbieżność między krzywymi -> overfitting w danym foldzie.

Wyniki: `figures\4-Krzywe_straty_per_fold.png`

In [ ]:
N_COLS = 6
N_ROWS = int(np.ceil(N_FOLDS / N_COLS))

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(N_COLS * 4, N_ROWS * 3.5), sharey=True)
fig.suptitle("Loss Curves per Fold", fontsize=14, fontweight="bold")
axes_flat = axes.flatten()

for i, (ax, history) in enumerate(zip(axes_flat, histories)):
    epochs = range(1, len(history.history["loss"]) + 1)
    best_e = np.argmin(history.history["val_smoothl1_loss"])
    ax.plot(epochs, history.history["smoothl1_loss"], color=FOLD_COLORS[i], lw=2, label="train")
    ax.plot(epochs, history.history["val_smoothl1_loss"], color=FOLD_COLORS[i], lw=2, ls="--", label="val")
    ax.axvline(best_e + 1, color="black", lw=1, ls=":", alpha=0.6, label=f"best (e{best_e+1})")
    ax.set_title(f"Fold {i+1}", fontweight="bold")
    ax.set_xlabel("Epoch")
    if i % N_COLS == 0:
        ax.set_ylabel("SmoothL1 Loss")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

for j in range(N_FOLDS, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig("./plots/cv_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### Wykres 2 - Pearson r i strata per fold

Słupki pokazują wynik walidacyjny dla każdego z 29 foldów.
Przerywana linia = średnia ± odchylenie standardowe.

Wyniki: `figures\4-Pearson_r_i_strata.png`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("29-Fold CV Performance Summary", fontsize=14, fontweight="bold")
folds_labels = [f"F{r['fold']}" for r in fold_results]

for ax, values, ylabel, title in [
    (axes[0], pearsons, "Pearson r", "Validation Pearson r per Fold"),
    (axes[1], losses, "SmoothL1 Loss", "Validation Loss per Fold"),
]:
    bars = ax.bar(folds_labels, values, color=FOLD_COLORS, edgecolor="white")
    mean_v, std_v = np.mean(values), np.std(values)
    ax.axhline(mean_v, color="black", lw=2, ls="--", label=f"mean = {mean_v:.3f} ± {std_v:.3f}")
    ax.fill_between(range(-1, N_FOLDS + 1), mean_v - std_v, mean_v + std_v, alpha=0.1, color="black")
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.002, f"{v:.2f}",
                ha="center", va="bottom", fontsize=6, rotation=90)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    ax.set_xticks(range(N_FOLDS))
    ax.set_xticklabels(folds_labels, rotation=45, ha="right", fontsize=8)
    if ylabel == "Pearson r":
        ax.set_ylim(-0.5, 1.15)

plt.tight_layout()
plt.savefig("./plots/cv_performance_summary.png", dpi=150, bbox_inches="tight")
plt.show()

### Wykres 3 - Sprawdzenie overfittingu

Każdy słupek pokazuje stratę treningową (kolor) + lukę do straty walidacyjnej (kreskowanie).
Mała luka = model dobrze generalizuje. Duża luka = model zapamiętał dane treningowe.

Wyniki: `figures\4-Sprawdzenie_overfittingu.png`

In [ ]:
fig, ax = plt.subplots(figsize=(8, N_FOLDS * 0.45 + 2))

for i, history in enumerate(histories):
    best_e = np.argmin(history.history["val_smoothl1_loss"])
    train_loss = history.history["smoothl1_loss"][best_e]
    val_loss = history.history["val_smoothl1_loss"][best_e]
    gap = val_loss - train_loss

    ax.barh(f"Fold {i+1}", train_loss, color=FOLD_COLORS[i], alpha=0.8,
            label="train" if i == 0 else "")
    ax.barh(f"Fold {i+1}", gap, left=train_loss, color=FOLD_COLORS[i],
            alpha=0.3, hatch="///", label="overfit gap" if i == 0 else "")
    ax.text(val_loss + 0.001, i, f"gap={gap:.4f}", va="center", fontsize=7)

ax.set_xlabel("SmoothL1 Loss at Best Epoch")
ax.set_title("Train vs Val Loss Gap (Overfitting Check)", fontweight="bold")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("./cv_overfit_check.png", dpi=150, bbox_inches="tight")
plt.show()

### Wykres 4 - Pearson r w czasie epok 

Każda linia to jeden fold. Pionowe kropkowane linie = najlepsza epoka danego foldu.
Duże rozproszenie krzywych świadczy o dużej wariancji między podmiotami.

Wyniki: `figures\4-Pearson_r_w_czasie epok.png`

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for i, history in enumerate(histories):
    epochs = range(1, len(history.history["val_pearson_r"]) + 1)
    best_e = np.argmin(history.history["val_smoothl1_loss"])
    ax.plot(epochs, history.history["val_pearson_r"],
            color=FOLD_COLORS[i], lw=1.5, alpha=0.8, label=f"F{i+1}")
    ax.axvline(best_e + 1, color=FOLD_COLORS[i], lw=1, ls=":", alpha=0.4)

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Pearson r")
ax.set_title("Pearson r Over Training — All Folds", fontweight="bold")
ax.legend(ncol=5, fontsize=7, loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("./cv_pearson_curves.png", dpi=150, bbox_inches="tight")
plt.show()